# 02 — Used Car Price Predictor: Model Training and Evaluation

This notebook turns the cleaned dataset from `01_eda_modeling.ipynb` into a working machine learning model.

The goal is not to test every possible model. The goal is to build a clean, reproducible modeling workflow that compares a few important regression models fairly.

## Project objective

Predict used-car listing prices from vehicle features such as make, model, year, mileage, fuel type, transmission, drivetrain, engine size, and location.

## Main question

Which regression model gives the best balance of prediction accuracy, reliability, and deployability for a used-car price prediction app?

## Modeling task

This is a **supervised regression** problem because the target variable, `price`, is numerical.

## Candidate models

This notebook compares:

- Dummy Regressor baseline
- Linear Regression
- Ridge Regression
- Decision Tree Regressor
- Random Forest Regressor
- Extra Trees Regressor
- XGBoost Regressor, if installed

## Main evaluation metrics

- **MAE**: average dollar error
- **RMSE**: penalizes larger mistakes more heavily
- **R²**: amount of price variation explained by the model

For this project, **MAE is the primary metric** because it is easiest to interpret in real pricing terms.

## Final output

The final model will be saved as:

```text
../models/car_price_pipeline.joblib
```

This saved pipeline can later be loaded by a Django app to make price predictions.

# 02 — Used Car Price Predictor: Model Training and Evaluation

1. Imports

2. Load processed dataset

3. Feature engineering

4. Define target and predictor variables

5. Train/test split

6. Preprocessing pipeline

7. Baseline model

8. Model training and comparison

9. Select best model

10. Actual vs predicted plot

11. Residual analysis

12. Save final model pipeline

13. Notebook conclusion

## Load Dataset

We will perform imputation after splitting our dataset into testing and training sets in order to avoid data leakage. So we perform feature engineering on `cleaned.csv`

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/cleaned.csv", low_memory=False)

df = df.replace({pd.NA: np.nan})
df.dtypes

id                    str
vin                   str
price             float64
miles             float64
stock_no              str
year              float64
make                  str
model                 str
trim                  str
body_type             str
vehicle_type          str
drivetrain            str
transmission          str
fuel_type             str
engine_size       float64
engine_block          str
seller_name           str
street                str
city                  str
state                 str
zip                   str
miles_was_zero      int64
dtype: object

### Performing Feature Engineering
Determining which features should be included in the training and the testing set

In [86]:
from datetime import datetime

# Dynamically fetch the current calendar year (e.g., 2026, 2027, etc.)
CURRENT_YEAR = datetime.now().year

df_model = df.copy()

# 1. Car age column
if "year" in df_model.columns:
    df_model["car_age"] = CURRENT_YEAR - df_model["year"]
    
    # Safety Guardrail: If data has a typo (e.g., year 2030), set age to NaN 
    # instead of a negative number, allowing the imputer to handle it safely.
    df_model.loc[df_model["car_age"] < 0, "car_age"] = np.nan

# 2. Miles per year column
if {"miles", "car_age"}.issubset(df_model.columns):
    # Instead of replacing 0 with 1 (which slightly distorts data for brand new cars),
    # calculate miles_per_year only where car_age > 0. If it's 0, leave it as NaN.
    df_model["miles_per_year"] = np.where(
        df_model["car_age"] > 0, 
        df_model["miles"] / df_model["car_age"], 
        np.nan
    )

# 3. Model Trim column
if {"model", "trim"}.issubset(df_model.columns):
    # Enforce string type, lowercasing, and whitespace stripping to prevent syntax duplicates
    clean_model = df_model["model"].astype("str").fillna("unknown").str.lower().str.strip()
    clean_trim = df_model["trim"].astype("str").fillna("unknown").str.lower().str.strip()
    
    df_model["model_trim"] = clean_model + " " + clean_trim

# 4. Region extraction from ZIP/Postal code
if "zip" in df_model.columns:
    df_model["region"] = df_model["zip"].astype("str").str.strip().str[:3]

df_model.shape

(358434, 26)

In [87]:
drop_cols = [
    "id",
    "vin",
    "seller_name",
    "street",
    "stock_no",
    "zip"
]

for cols in drop_cols:
    if cols in df_model.columns:
        df_model.drop(columns=cols, inplace=True)

df_model.shape

(358434, 20)

In [88]:
df_model.head()

,price,miles,year,make,model,trim,body_type,vehicle_type,drivetrain,transmission,fuel_type,engine_size,engine_block,city,state,miles_was_zero,car_age,miles_per_year,model_trim,region
0,179999.0,9966.0,2017.0,acura,nsx,base,coupe,car,4wd,automatic,electric / premium unleaded,3.5,v,edmundston,nb,0,9.0,1107.333333,nsx base,e3v
1,179995.0,5988.0,2017.0,acura,nsx,base,coupe,car,4wd,automatic,electric / premium unleaded,3.5,v,notre-dame-des-pins,qc,0,9.0,665.333333,nsx base,g0m
2,168528.0,24242.0,2017.0,acura,nsx,base,coupe,car,4wd,automatic,electric / premium unleaded,3.5,v,coquitlam,bc,0,9.0,2693.555556,nsx base,v3k
3,220000.0,6637.0,2020.0,acura,nsx,base,coupe,car,4wd,automatic,electric / premium unleaded,3.5,v,pickering,on,0,6.0,1106.166667,nsx base,l1v
4,220000.0,6637.0,2020.0,acura,nsx,base,coupe,car,4wd,automatic,electric / premium unleaded,3.5,v,pickering,on,0,6.0,1106.166667,nsx base,l1v


## Define Target and Predictor variables

In [89]:
from sklearn.model_selection import train_test_split

TARGET = "price"

X = df_model.drop(columns=[TARGET])
y = df_model.drop(columns=X.columns)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (358434, 19)
y shape: (358434, 1)


## Train Test Split

In [90]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42)

print("Number of rows in X_train:", X_train.shape[0])
print("Number of rows in y_test:", y_test.shape[0])

df_model.info()

Number of rows in X_train: 286747
Number of rows in y_test: 71687
<class 'pandas.DataFrame'>
RangeIndex: 358434 entries, 0 to 358433
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   price           358434 non-null  float64
 1   miles           335025 non-null  float64
 2   year            358419 non-null  float64
 3   make            358434 non-null  str    
 4   model           354245 non-null  str    
 5   trim            325115 non-null  str    
 6   body_type       327710 non-null  str    
 7   vehicle_type    323765 non-null  str    
 8   drivetrain      325145 non-null  str    
 9   transmission    327945 non-null  str    
 10  fuel_type       295953 non-null  str    
 11  engine_size     294138 non-null  float64
 12  engine_block    293740 non-null  str    
 13  city            351849 non-null  str    
 14  state           351789 non-null  str    
 15  miles_was_zero  358434 non-null  int64  
 16  c

## Preprocessing Pipeline 

Numeric columns are imputed with the median, and Categorical columns are filled with `unknown` and one-hot encoded.

Use `ColumnTransformer` later to combine these two pipelines inside one reproducible preprocessing object.

In [91]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['miles', 'year', 'engine_size', 'miles_was_zero', 'car_age', 'miles_per_year']
Categorical columns: ['make', 'model', 'trim', 'body_type', 'vehicle_type', 'drivetrain', 'transmission', 'fuel_type', 'engine_block', 'city', 'state', 'model_trim', 'region']


In [92]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]
)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
        ("numerical", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols)
])

## Evaluation Function

We will use MAE, RMSE and R².

In [93]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression_model(model, X_test, y_test):
    prediction = model.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)
    rmse = mean_squared_error(y_test, prediction)
    r2 = r2_score(y_test, prediction)

    return {
        "MAE" : mae,
        "RMSE" : rmse,
        "R2" : r2
    }

## Baseline Model 

Before training any models, I create a baseline using DummyRegressor.

This model simply predicts the median price every time. I expect any of the other models to beat this.

In [95]:
from sklearn.dummy import DummyRegressor

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="median"))
])

baseline_pipeline.fit(X_train, y_train)

evaluate_regression_model(baseline_pipeline, X_test, y_test)

{'MAE': 11661.111163809337,
 'RMSE': 427176860.5074839,
 'R2': -0.03999402911186789}